# Is the leak-free network running DD?

**Jobs `50075339` (round-0 interventions), `50077245` (chaining interventions) and `50082120` (withholding interventions), each three-armed.**

Every previous answer to this question in this project came from a decoding score. Finding 8's `next_node` probe correlated at Spearman +0.96 with solve_rate and was then retired outright when a hand rule that does no deduction at all (`min_unknown`, 0.632) beat the model's own features (0.545). So this run uses interventions instead.

## The eval

DD's step is a **rule** plus a **guard**: eliminate the colours of a node's known neighbours, and emit a node *only when* that leaves exactly one colour. The three tests take those apart.

Take a node `v` that DD forces in some round: its known neighbours already use `q-1 = 3` of the 4 colours, and **exactly one** of them, `w`, holds the colour `a` that is doing the work, so `v` must take the one remaining colour `c`.

- **Round 0 — the rule on given facts** (`sudoku_circuits.py`). Recolour `w` to `c` — **a single token**, asserted by `build_pair` as `(clean != corrupt).sum() == 1`. Now `a` is free and DD's answer for `v` changes from `c` to `a`. Does the prediction follow? `w` is a **hint**, given in the prompt.
- **Chained — the rule on facts it produced** (`sudoku_chain.py`). The same edit, but `w` was **deduced by the model** in an earlier round, so the edited token sits in the reasoning section. This is propagation, not a single rule application.
- **Withheld — the guard** (`sudoku_skip.py`). Recolour `w` to a colour that **another** neighbour already holds. Now the freed colour is held by nobody, the forcing set covers only `q-2` colours, `v` has two candidates and **DD does not emit `v` in this round at all**. The measurement moves one token left: from the logits that predict `v`'s colour to the logits that predict the node identity. Does the model stop naming `v`? Every case carries its own **placebo** — the same kind of one-token edit on a hint that is *not* a neighbour of `v`, which leaves `v` forced and DD's round unchanged. The placebo is the ceiling the withhold rate is read against; without it a low rate would only say "edits perturb the node stream".

Chance on the rule tests is `1/n_colors = 0.25`. None of this can be passed by matching the prompt: the paired prompts differ in one token and the correct answers differ.

## Headline

**YES — it runs DD's inference step: the rule, the chaining, and the guard.** The leak-free 1.0000 model applies the elimination rule on **350/350** round-0 counterfactuals, chains it off its own deductions at **0.980** with no decay from round 1 to round 4, and when the forcing is *broken* rather than redirected it stops naming the node on **350/350** against a **0.940** placebo ceiling. The one part of DD it does not reproduce is the within-round emission *order* (ascending node index) — which is DD's own convention anyway, and the guard test shows that *which* nodes it emits does track determinacy.

The guard is also where the two leak-free checkpoints finally differ: the 0.8540 model withholds only **0.824** against the 1.0000 model's **1.000**, paired, **p = 1.1e-13**.

In [ ]:
import json, os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

SCRATCH = os.environ["SCRATCH"]
CIRC = f"{SCRATCH}/spin_gnn/sudoku_circuits_leakfree_2026-10-28"
CHAIN = f"{SCRATCH}/spin_gnn/sudoku_chain_2026-10-28"
SKIP = f"{SCRATCH}/spin_gnn/sudoku_skip_2026-10-28"
ARMS = [("leak-free 1.0000", "A_leakfree_1000"),
        ("leak-free 0.8540", "B_leakfree_0854"),
        ("leaked 0.9960", "C_leaked_0996")]
CHANCE = 0.25
SITES = ["edited_hint", "graph_row_u", "graph_row_w", "predict_pos"]

beh = {k: pd.read_csv(f"{CIRC}/{d}/behavior.csv") for k, d in ARMS}
patch = {k: pd.read_csv(f"{CIRC}/{d}/patching.csv") for k, d in ARMS}
heads = {k: pd.read_csv(f"{CIRC}/{d}/head_ablation.csv") for k, d in ARMS}
chain, skip, skip_beh = {}, {}, {}
for k, d in ARMS:
    with open(f"{CHAIN}/{d}/chain_summary.json") as f:
        chain[k] = json.load(f)
    with open(f"{SKIP}/{d}/skip_summary.json") as f:
        skip[k] = json.load(f)
    skip_beh[k] = pd.read_csv(f"{SKIP}/{d}/skip_behavior.csv")
print({k: (len(beh[k]), chain[k]["n_cases"], skip[k]["n_cases"]) for k, _ in ARMS})

## Metrics table

`clean` is accuracy on the unmodified prompt. A high `clean` with a low `flipped` means the model is right for a reason other than the elimination — which is exactly the leaked arm's profile. `reverts_to_original` is the "detected the edited prefix" signature and only applies to the chained arm.

In [ ]:
rows = []
for k, _ in ARMS:
    b, c, s = beh[k], chain[k], skip[k]
    rows.append({"arm": k, "test": "round 0 (hint)", "n": len(b),
                 "clean": b.clean_correct.mean(), "moved": b.corrupt_flipped.mean(),
                 "reverts": np.nan, "placebo": np.nan,
                 "logit_clean": b.clean_logit_diff.mean(),
                 "logit_corrupt": b.corrupt_logit_diff.mean()})
    rows.append({"arm": k, "test": "chained (deduced)", "n": c["n_cases"],
                 "clean": c["clean_correct"], "moved": c["corrupt_flipped"],
                 "reverts": c["reverts_to_original"], "placebo": np.nan,
                 "logit_clean": np.nan, "logit_corrupt": np.nan})
    rows.append({"arm": k, "test": "withheld (guard)", "n": s["n_cases"],
                 "clean": s["clean_emits_v"], "moved": 1.0 - s["corrupt_emits_v"],
                 "reverts": np.nan, "placebo": s["placebo_emits_v"],
                 "logit_clean": np.nan, "logit_corrupt": np.nan})
metrics = pd.DataFrame(rows)
metrics["verdict"] = np.where(metrics.moved >= 0.60, "follows DD", "does not")
display(metrics.round(3))

print("`moved` = flipped to DD's new colour (rule tests) / stopped naming v (guard test).")
print("`placebo` = still names v after a one-token edit that leaves v forced (guard ceiling).")

## Key plot 1 — the rule holds on given facts *and* on facts it derived itself

The right panel is the chaining claim. A model that chained one step but not three would show round 1 high and the later rounds falling off. It does not happen; if anything the rate rises with depth. The deep bins are small because deeply chained forcings are rare, not because anything was filtered.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
x = np.arange(len(ARMS))
r0 = [beh[k].corrupt_flipped.mean() for k, _ in ARMS]
ch = [chain[k]["corrupt_flipped"] for k, _ in ARMS]
cols = ["#1f77b4", "#2ca02c", "#d62728"]
ax1.bar(x - 0.2, r0, 0.38, color=cols, edgecolor="k", label="round 0 (hint)")
ax1.bar(x + 0.2, ch, 0.38, color=cols, edgecolor="k", hatch="///", alpha=0.75,
        label="chained (deduced)")
ax1.axhline(CHANCE, color="k", ls="--")
ax1.set_xticks(x); ax1.set_xticklabels([k for k, _ in ARMS], fontsize=9)
ax1.set_ylabel("flipped to DD's new answer"); ax1.set_ylim(0, 1.1)
ax1.legend(fontsize=9)

br = chain["leak-free 1.0000"]["by_round"]
ks = sorted(int(k) for k in br)
ax2.plot(ks, [br[str(k)]["flipped"] for k in ks], "o-", lw=3, ms=11, color="#1f77b4")
ax2.axhline(CHANCE, color="k", ls="--")
ax2.set_xlabel("DD round the node was deduced in"); ax2.set_xticks(ks)
ax2.set_ylabel("flipped"); ax2.set_ylim(0, 1.1)
ax2.set_title("no decay with depth (leak-free 1.0000)")
for k in ks:
    ax2.annotate(f"n={br[str(k)]['n']}", (k, br[str(k)]["flipped"]),
                 textcoords="offset points", xytext=(0, -26), ha="center", fontsize=9)
plt.tight_layout(); plt.show()

## Key plot 1b — the guard: break the forcing and it stops naming the node

This is the converse of the two tests above, and it is the one a referee reaches for first. Both rule tests *select* nodes that DD forces and then *redirect* the forcing, so a model that simply wrote a colour for whichever unknown node came next in index order would pass them. Here the forcing is **removed**: `v` goes from one candidate colour to two, DD drops it from the round, and the question is whether the model drops it too.

The left panel reads three bars per arm against each other: `clean` (names `v`, as DD does), `placebo` (a one-token edit elsewhere that leaves `v` forced — the ceiling, since any one-token edit perturbs the stream a little), and `broken` (the forcing removed). The right panel asks what it says *instead* when it stops naming `v`.

**The paired contrast at the bottom is the find of this notebook.** The three arms are generated from the same instance stream with the same seed, so case *i* is the same graph, the same node and the same edit in every arm, and arms A and B can be compared instance-for-instance with McNemar's exact test.

In [ ]:
from scipy.stats import binomtest

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
x = np.arange(len(ARMS))
w = 0.26
for i, (lab, key, hatch) in enumerate([("clean (names v, as DD does)", "clean_emits_v", ""),
                                       ("placebo edit (v still forced)", "placebo_emits_v", ".."),
                                       ("forcing broken (DD drops v)", "corrupt_emits_v", "//")]):
    ax1.bar(x + (i - 1) * w, [skip[k][key] for k, _ in ARMS], w,
            color=["#9ecae1", "#9e9ac8", "#d62728"][i], edgecolor="k", hatch=hatch, label=lab)
ax1.set_xticks(x); ax1.set_xticklabels([k for k, _ in ARMS], fontsize=9)
ax1.set_ylabel("fraction of cases that still name v"); ax1.set_ylim(0, 1.25)
ax1.legend(fontsize=8, loc="upper left", ncol=1, framealpha=0.95)
ax1.set_title("the guard: it names v unless DD would not")
for i, key in enumerate(["clean_emits_v", "placebo_emits_v", "corrupt_emits_v"]):
    for xi, (k, _) in enumerate(ARMS):
        ax1.annotate(f"{skip[k][key]:.3f}", (xi + (i - 1) * w, skip[k][key]),
                     textcoords="offset points", xytext=(0, 3), ha="center",
                     fontsize=7, rotation=90)

a = skip["leak-free 1.0000"]
n_skip = a["n_cases"] * (1 - a["corrupt_emits_v"])
names_node = a["skipped_names_a_node"] * n_skip
parts = [("a node DD now forces", names_node * a["skipped_forced_given_node"]),
         ("a node not forced", names_node * (1 - a["skipped_forced_given_node"])),
         ("BEGIN_ROUND\n(DD's own answer when\nnothing is determined)", a["skipped_non_node_tokens"].get("BEGIN_ROUND", 0))]
ax2.barh([p[0] for p in parts], [p[1] for p in parts],
         color=["#2ca02c", "#ff7f0e", "#1f77b4"], edgecolor="k")
for i, (_, v) in enumerate(parts):
    ax2.annotate(f"{v:.0f}  ({v / n_skip:.1%})", (v, i), textcoords="offset points",
                 xytext=(6, 0), va="center", fontsize=10)
ax2.set_xlabel(f"cases (of the {n_skip:.0f} where it stopped naming v)")
ax2.set_title("what the leak-free 1.0000 model names instead")
ax2.set_xlim(0, n_skip * 1.35)
plt.tight_layout(); plt.show()

# no decay with v's position in the round
print("broken-edit rate by v's rank within DD round 0 (leak-free 1.0000):")
for r, d in sorted(a["by_round0_rank"].items()):
    print(f"  rank {r}: still names v {d['broken']:.3f}   placebo {d['placebo']:.3f}   n={d['n']}")

# paired McNemar, A vs B and A vs C, on the identical first 250 cases
A = skip_beh["leak-free 1.0000"]
for other in ["leak-free 0.8540", "leaked 0.9960"]:
    O = skip_beh[other]
    m = min(len(A), len(O))
    key = ["node", "round0_rank", "binding_nbr", "n_known"]
    assert A[key].head(m).reset_index(drop=True).equals(O[key].reset_index(drop=True)), \
        "case lists differ -- not paired"
    a_emit = A.corrupt_emits_v.head(m).to_numpy()
    o_emit = O.corrupt_emits_v.to_numpy()
    b = int(((~a_emit) & o_emit).sum())   # A withholds, other names v
    c = int((a_emit & (~o_emit)).sum())   # other withholds, A names v
    p = binomtest(b, b + c, 0.5).pvalue if b + c else 1.0
    print(f"\nleak-free 1.0000 vs {other}  (n={m} identical cases)")
    print(f"  still names v: {a_emit.mean():.3f} vs {o_emit.mean():.3f}")
    print(f"  discordant {b}-{c}, McNemar exact p = {p:.3g}")
    ok = A.clean_emits_v.head(m).to_numpy() & O.clean_emits_v.to_numpy()
    print(f"  restricted to cases both get right clean (n={ok.sum()}): "
          f"{a_emit[ok].mean():.3f} vs {o_emit[ok].mean():.3f}")

## Key plot 2 — where the recoloured fact travels

`recovery` is how much of the flip is restored by patching the clean activation at that (layer, site). **Pre-registration (c) was falsified**: `graph_row_u` and `graph_row_w` recover 0.000 at every layer in all three arms, so the deduction never routes through the adjacency table rows. L11 at the prediction position is the built-in positive control and must read 1.00.

In [ ]:
for k, _ in ARMS:
    m = patch[k].groupby(["layer", "site"]).recovery.mean().unstack()[SITES]
    print(f"\n=== {k} ===")
    display(m.round(3))

## Comparison to the previous best answer on this question

There is no previous *causal* measurement of the leak-free task — it had never been run, which is why this job exists. The comparison that does exist is **finding 2** of this project ("the model is not running DD"), which rested on probes, an algorithm-ID test, a counterfactual and an SAE, all measured on the **leaked** 0.9960 checkpoint. That counterfactual is reproduced here as arm C.

| | finding 2 (leaked 0.9960) | this run (leak-free 1.0000) |
|---|---|---|
| round-0 flip rate | 0.388 (≈ chance 0.25) | **1.000** |
| chained flip rate | 0.008 | **0.980** |
| reverts to original, chained | 0.992 | 0.020 |
| withholds when forcing broken | 0.992 | **1.000** (placebo ceiling 0.940) |

**Finding 2 stands for the leaked model and is overturned for the leak-free one.** The leaked model's chained arm is the sharpest result in the set: shown an edited deduction it keeps its own answer 99.2% of the time, which is precisely what a model reading finding 16's adjacency-order leak would do, since the graph row still encodes the original colouring.

The guard row is the **falsified** pre-registration (c) of job 50082120: I predicted the leaked model would *keep* emitting `v`, on the reasoning that its adjacency rows still encode the original colouring so nothing can be un-determined for it. It withholds as well as the leak-free model does. So the guard is **not** part of what the leak replaced — the leaked twin learned which nodes are determined and never learned the rule that says what colour they take.

**And pre-registration (d) was falsified twice, in opposite directions.** On the rule tests I predicted A > B > C monotone in solve_rate and got A = B ≫ C, leaving open what the last 0.146 of solve_rate buys. The guard test answers it: arm B withholds only **0.824** against arm A's **1.000**. Because the arms share an instance stream, that is a paired comparison — **0–44 discordant, McNemar exact p = 1.1e-13**, and 0.000 vs 0.188 if restricted to cases both arms get right on the clean prompt — while A vs C is a tie (0–2, p = 0.5). This is the first measurement in the project that resolves the two leak-free checkpoints at all, and it puts their difference in **the guard, not the elimination rule**.

In [ ]:
## What it does *not* do

**It does not choose the node order by deduction.** Events are emitted in **ascending node index for 100% of rounds** (`scripts/order_baseline_next_node.py`), and on the same max-over-v F1, a hand rule that does no deduction at all — `min_unknown`, "smallest index above the previous whose colour is unknown" — scores **0.632** against the model's own `next_node` features at **0.545**. Finding 24 adds the training-side control: permuting event order within rounds costs nothing resolvable (−0.0147, p=0.225).

Two qualifications now attach to that negative, both from the guard test. Ascending node index **is DD's own convention** in this implementation — the reference solver emits each round's determined nodes in index order too — so matching it is not evidence of a shortcut, and departing from it would not be evidence of DD. And the guard test shows that *which* nodes it names does track determinacy: break the forcing and the node it names instead is one that is genuinely forced under the edited hints 97.6% of the time, or `BEGIN_ROUND` when nothing is.

**It does not route the deduction through the adjacency table.** `graph_row_u` and `graph_row_w` recover 0.000 at every layer in all three arms.

So the working picture is: **sweep nodes in index order; emit one only when its known neighbours leave it a single colour, counting the ones it just derived.** That is DD's propagation step. What is not demonstrated is a frontier data structure — only behaviour consistent with recomputing determinacy per node.

## Comparison to the previous best answer on this question

There is no previous *causal* measurement of the leak-free task — it had never been run, which is why this job exists. The comparison that does exist is **finding 2** of this project ("the model is not running DD"), which rested on probes, an algorithm-ID test, a counterfactual and an SAE, all measured on the **leaked** 0.9960 checkpoint. That counterfactual is reproduced here as arm C.

| | finding 2 (leaked 0.9960) | this run (leak-free 1.0000) |
|---|---|---|
| round-0 flip rate | 0.388 (≈ chance 0.25) | **1.000** |
| chained flip rate | 0.008 | **0.980** |
| reverts to original, chained | 0.992 | 0.020 |

**Finding 2 stands for the leaked model and is overturned for the leak-free one.** The leaked model's chained arm is the sharpest result in the set: shown an edited deduction it keeps its own answer 99.2% of the time, which is precisely what a model reading finding 16's adjacency-order leak would do, since the graph row still encodes the original colouring.

**Pre-registration (d) was falsified informatively.** I predicted A > B > C monotone in solve_rate. It is A = B ≫ C: the 0.8540 leak-free model applies *and* chains the rule exactly as well as the 1.0000 one, so the last 0.146 of solve_rate is bought by something other than the forcing rule. The real boundary is leak-free vs leaked.

import subprocess
print("git commit :", subprocess.run(["git", "rev-parse", "--short", "HEAD"],
                                     capture_output=True, text=True).stdout.strip())
print("jobs       : 50075339 (round 0, COMPLETED 00:05:53), 50077245 (chained, COMPLETED 00:01:52),")
print("             50082120 (withheld/guard, COMPLETED; first attempt 50080911 crashed, fixed in fb06610)")
print("sbatch     : scripts/slurm/sudoku2_circuits_leakfree.sbatch, sudoku2_chain.sbatch, sudoku2_skip.sbatch")
for k, _ in ARMS:
    c, s = chain[k], skip[k]
    print(f"{k:20s} seed_start={c['seed_start']} (rule) / {s['seed_start']} (guard)  "
          f"checkpoint={os.path.basename(c['checkpoint'])}")
print("\nconfig: N=24, degree=5, n_colors=4, rho=0.60/rho_max=0.95, d_model=256, L=12, ff=1024,")
print("        shuffle_adj=True for both leak-free arms (read from each checkpoint's own config).")
print("deck  : notebooks/sudoku_dd_verdict_2026-10-28/slides.pdf (python -m notebooks.make_dd_verdict_deck)")

## Caveats

1. **The chained arm's one weak direction.** Editing an emitted deduction makes the prefix inconsistent with the graph and hints, so a model that *detects* the edit and keeps its own answer would score as "does not chain". `reverts_to_original` separates those cases, and for the leak-free arms it is 0.008–0.020 — the misses are that benign case, not a failure to chain. A *high* flip rate carries no such ambiguity, which is why this direction of the result is the safe one.
2. **The guard's placebo ceiling is not 1.000** (0.900–0.964 across arms). Any one-token edit perturbs the node stream a little, so a few points of every withhold rate are edit perturbation rather than deduction. The claim is read as broken-vs-placebo *within* an arm (1.000 vs 0.940 in arm A, a +13.940 vs +0.950 logit drop on `v`), and the A-vs-B contrast is against the same ceiling in both arms. Had the placebo collapsed too, the test would have said nothing — that spoiler case was pre-registered.
3. **Forced-node cases only.** All three tests select nodes whose forcing set covers exactly `q-1` colours with a unique binding neighbour. That is DD's elimination rule by construction; the guard test is what keeps it from being circular, since it removes the forcing rather than redirecting it, but it still says nothing about instances where DD needs several rounds of bookkeeping to expose such a node.
4. **One seed, one checkpoint per arm**, and `n_colors = 4` so chance on the rule tests is 0.25.
5. **Round depth is capped by the data**: only 4 chained cases reach round 4, so "no decay" is well powered at rounds 1–2 and suggestive at 3–4. The guard test's rank breakdown is better balanced (n = 93/92/81/84 at ranks 0–3).
6. **The guard job's first attempt crashed** (`50080911`): 18 of 350 constructed cases were still forced after the edit, because a round-0 deduction emitted *before* `v` can hold the colour being freed. Forcing is now computed from everything the prefix reveals and each broken assignment is re-checked inside the case finder (`fb06610`), with the unique holder required to be a hint. That is a change to case *selection*, not to the measurement.

In [ ]:
import subprocess
print("git commit :", subprocess.run(["git", "rev-parse", "--short", "HEAD"],
                                     capture_output=True, text=True).stdout.strip())
print("jobs       : 50075339 (round 0, COMPLETED 00:05:53), 50077245 (chained, COMPLETED 00:01:52)")
print("sbatch     : scripts/slurm/sudoku2_circuits_leakfree.sbatch, scripts/slurm/sudoku2_chain.sbatch")
for k, _ in ARMS:
    c = chain[k]
    print(f"{k:20s} seed_start={c['seed_start']}  checkpoint={os.path.basename(c['checkpoint'])}")
print("\nconfig: N=24, degree=5, n_colors=4, rho=0.60/rho_max=0.95, d_model=256, L=12, ff=1024,")
print("        shuffle_adj=True for both leak-free arms (read from each checkpoint's own config).")
print("deck  : notebooks/sudoku_dd_verdict_2026-10-28/slides.pdf (python -m notebooks.make_dd_verdict_deck)")

## Caveats

1. **The chained arm's one weak direction.** Editing an emitted deduction makes the prefix inconsistent with the graph and hints, so a model that *detects* the edit and keeps its own answer would score as "does not chain". `reverts_to_original` separates those cases, and for the leak-free arms it is 0.008–0.020 — the misses are that benign case, not a failure to chain. A *high* flip rate carries no such ambiguity, which is why this direction of the result is the safe one.
2. **Forced-node cases only.** Both tests select nodes whose forcing set covers exactly `q-1` colours with a unique binding neighbour. That is DD's elimination rule by construction; it says nothing about instances where DD needs more than one round of bookkeeping to expose such a node.
3. **One seed, one checkpoint per arm**, and `n_colors = 4` so chance is 0.25.
4. **Round depth is capped by the data**: only 4 cases reach round 4, so "no decay" is well powered at rounds 1–2 and suggestive at 3–4.